# Sample v3: 500 suspended / 500 active, edit-adjusted first mention

Builds on `sample_1000_v2_activity_before_accusation.ipynb`. The TFA CSV stores each post's text as retrieved, including
edits, so a name added in an edit is dated too early by the post's `created_utc`. Only each account's **first** accusing post
is stored (`first_post_id`), and Reddit keeps only the **last** edit time, so:

- **`first_mention_adj_utc`** = the later of the first accusing post's `created_utc` and its `edited` time;
- accounts whose first accusing post was last edited **more than 48 hours** after it was created are dropped (the date is too uncertain).

**Eligible** = all of:
1. in `accused_accounts_not_deleted.parquet` (PRAW status active or suspended);
2. has an Arctic Shift history pull marked `complete` in `arctic_pull_done.parquet`;
3. **first action after r/TheseFuckingAccounts existed** (2014-12-21 00:03:16 UTC), first action = earliest of Arctic Shift's `earliest_post_at`, `earliest_comment_at` and the oldest stored post or comment;
4. **first accusing post not edited more than 48 hours after creation**;
5. **first accused before 30 November 2022**: `first_mention_adj_utc` < 2022-11-30 00:00:00 UTC;
6. **first action before 30 November 2022**;
7. **first action before first accusation**: `first_action_utc` < `first_mention_adj_utc`.

Using the edit time can make an account eligible that v2 dropped (first action between post creation and an edit within 48 h).

**Resampling**, per status, so the result is a simple random sample of the v3 pool while keeping as many v2 accounts as possible:
the v3 pool splits into accounts that were in the v2 pool (A) and newly eligible ones (B). The number of slots for B is drawn
from the hypergeometric distribution (its share under a uniform draw); B's slots are a random draw from B; A's slots are filled
with the v2 sample accounts still in A, topped up at random from the rest of A (or subsampled at random if there are too many).
`SEED = 2026` throughout.

Inputs are read-only; outputs go to `sample_1000_v3_output/` and `save_new()` refuses to overwrite.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
import glob
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

# Folders, sample size, random seed and the cut-off times (subreddit creation; 30 Nov 2022).
DATA_DIR = os.path.join('..', 'p2_data_cleaning', 'data')
OUT_DIR = 'sample_1000_v3_output'
PREV_DIR = 'sample_1000_v2_output'
TFA_CSV = '/data/Emily_data/TheseFuckingAccounts_submissions.csv'
MAX_EDIT_LAG_H = 48
N_PER_GROUP = 500
SEED = 2026
TFA_CREATED_UTC = 1419120196                                                  # 2014-12-21 00:03:16 UTC, PRAW subreddit.created_utc
ACCUSED_BEFORE_UTC = int(pd.Timestamp('2022-11-30', tz='UTC').timestamp())   # 1669766400
os.makedirs(OUT_DIR, exist_ok=True)

# Write a parquet file only if it does not exist yet (never overwrite a saved sample).
def save_new(df, name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        print(f'exists, not overwritten: {path}')
        return
    df.to_parquet(path, engine='pyarrow')
    print(f'wrote {path}  {df.shape}')

# Read the stored Arctic Shift history (kind = 'posts' or 'comments'), optionally only some columns/accounts.
def read_history(kind, columns=None, keys=None):
    # file by file: some parts store all-null columns as type null, which breaks a unified dataset schema.
    # repair-* parts re-pulled whole histories, so rows repeat across files -> dedupe on (outer_key, id)
    filters = [('outer_key', 'in', list(keys))] if keys is not None else None
    parts = [pd.read_parquet(f, engine='pyarrow', columns=columns, filters=filters)
             for f in sorted(glob.glob(os.path.join(DATA_DIR, f'arctic_{kind}', '*.parquet')))]
    df = pd.concat([p for p in parts if len(p)], ignore_index=True)
    return df.drop_duplicates(['outer_key', 'id']).reset_index(drop=True)

## 1. Eligible pool

In [ ]:
# Accused accounts (not deleted), the history-pull ledger, and Arctic Shift account metadata.
accused = pd.read_parquet(os.path.join(DATA_DIR, 'accused_accounts_not_deleted.parquet'))
pull_done = pd.read_parquet(os.path.join(DATA_DIR, 'arctic_pull_done.parquet'))
meta = pd.read_parquet(os.path.join(DATA_DIR, 'account_meta_arctic.parquet'))
assert pull_done['username_key'].is_unique
print('duplicate meta keys (kept last):', meta['username_key'].duplicated().sum())
meta = meta.drop_duplicates('username_key', keep='last')

# Earliest stored post or comment per account.
hist_first = pd.concat([read_history('posts', ['outer_key', 'id', 'created_utc']),
                        read_history('comments', ['outer_key', 'id', 'created_utc'])]
                       ).groupby('outer_key')['created_utc'].min()

# One row per accused account, with pull status and the archive's first-post/first-comment times.
pool = (accused.merge(pull_done, on='username_key', how='left', validate='one_to_one')
               .merge(meta[['username_key', 'earliest_post_at', 'earliest_comment_at']],
                      on='username_key', how='left', validate='one_to_one'))
pool['history_first_utc'] = pool['username_key'].map(hist_first)
# First action = earliest of the archive's first post, first comment, and first stored row.
pool['first_action_utc'] = pool[['earliest_post_at', 'earliest_comment_at', 'history_first_utc']].min(axis=1)
pool['complete'] = pool['complete'].eq(True)          # accounts with no ledger row are not complete

# edit time of the first accusing post: CSV `edited` is the last edit's epoch or 'False'. The 2023-2026 Arctic corpus has no
# `edited` column, but every account first accused before the cutoff was first accused in a CSV post (asserted below).
tfa = pd.read_csv(TFA_CSV, low_memory=False, usecols=['id', 'created_utc', 'edited'])
assert tfa.groupby('id')['edited'].nunique().le(1).all()
edited_utc = pd.to_numeric(tfa.drop_duplicates('id').set_index('id')['edited'], errors='coerce')
pool['accusation_post_edited_utc'] = pool['first_post_id'].map(edited_utc)
pool['edit_lag_h'] = (pool['accusation_post_edited_utc'] - pool['first_mention_utc']) / 3600
assert not (pool['edit_lag_h'] < 0).any()
# Adjusted first mention = the later of the post's creation and its last edit, since the name may have been added by the edit.
pool['first_mention_adj_utc'] = pool[['first_mention_utc', 'accusation_post_edited_utc']].max(axis=1)

# Eligibility funnel: each rule applied in turn; the table counts accounts by status after each step.
steps = {'accused, not deleted': pd.Series(True, index=pool.index)}
steps['+ complete Arctic history'] = steps['accused, not deleted'] & pool['complete']
steps['+ first action after TFA created'] = steps['+ complete Arctic history'] & (pool['first_action_utc'] > TFA_CREATED_UTC)
# Drop accounts whose accusing post was edited more than 48 h after it was posted.
steps['+ accusing post not edited >48h later'] = steps['+ first action after TFA created'] & ~(pool['edit_lag_h'] > MAX_EDIT_LAG_H)
steps['+ first accused before 2022-11-30'] = steps['+ accusing post not edited >48h later'] & (pool['first_mention_adj_utc'] < ACCUSED_BEFORE_UTC)
steps['+ first action before 2022-11-30'] = steps['+ first accused before 2022-11-30'] & (pool['first_action_utc'] < ACCUSED_BEFORE_UTC)
steps['+ first action before first accused'] = steps['+ first action before 2022-11-30'] & (pool['first_action_utc'] < pool['first_mention_adj_utc'])
funnel = pd.DataFrame({k: pool.loc[v, 'account_status'].value_counts() for k, v in steps.items()}).T
funnel['total'] = funnel.sum(axis=1)
funnel

In [ ]:
# Eligible pool = accounts that pass every rule. Shows how many histories are cut at the 10k-item cap,
# and median activity by status.
eligible = pool[steps['+ first action before first accused']].reset_index(drop=True)
assert eligible['first_post_id'].isin(set(tfa['id'])).all(), 'an eligible account was first accused outside the CSV'
# Flag accounts that were not in the v2 pool (newly eligible after the edit adjustment).
prev_pool = pd.read_parquet(os.path.join(PREV_DIR, 'eligible_pool.parquet'))
eligible['in_v2_pool'] = eligible['username_key'].isin(prev_pool['username_key'])
print('newly eligible (not in v2 pool):', (~eligible['in_v2_pool']).groupby(eligible['account_status']).sum().to_dict())
print('accusing post edited (within 48 h):', eligible['accusation_post_edited_utc'].notna().sum())
print('history truncated at the 10k cap:', eligible['history_truncated'].sum())
# Days from first action to the adjusted first accusation.
eligible['days_to_accused'] = (eligible['first_mention_adj_utc'] - eligible['first_action_utc']) / 86400
eligible.groupby('account_status')[['n_posts', 'n_comments', 'times_mentioned', 'days_to_accused']].median()

## 2. Draw the sample

In [ ]:
# Which v2 sampled accounts are no longer eligible, and the first rule each one fails.
prev_sample = pd.read_parquet(os.path.join(PREV_DIR, 'sample_accounts.parquet'), columns=['username_key', 'account_status'])
dropped = prev_sample[~prev_sample['username_key'].isin(eligible['username_key'])].copy()
failed = {k: set(pool.loc[~v, 'username_key']) for k, v in steps.items()}
dropped['reason'] = dropped['username_key'].map(lambda u: next(k for k in steps if u in failed[k]).lstrip('+ '))
print('v2 accounts dropped:'); print(dropped.groupby(['account_status', 'reason']).size().to_string())

# Draw so the result is a uniform random sample of the v3 pool while keeping as many v2 accounts as possible.
# A = accounts also in the v2 pool, B = newly eligible ones; the hypergeometric draw sets how many come from B.
rng = np.random.default_rng(SEED)
parts = []
for status, g in eligible.groupby('account_status'):
    A, B = g[g['in_v2_pool']], g[~g['in_v2_pool']]
    k_new = int(rng.hypergeometric(len(B), len(A), N_PER_GROUP))          # B's share of a uniform draw from A + B
    n_old = N_PER_GROUP - k_new
    # Reuse v2 sampled accounts first; if too few are still eligible, top up from the rest of A.
    kept = A[A['username_key'].isin(prev_sample['username_key'])]
    if len(kept) >= n_old:
        old_part = kept.sample(n=n_old, random_state=SEED)
    else:
        old_part = pd.concat([kept, A[~A['username_key'].isin(prev_sample['username_key'])].sample(n=n_old - len(kept), random_state=SEED)])
    new_part = B.sample(n=k_new, random_state=SEED)
    parts.append(pd.concat([old_part, new_part]))
    print(f'{status}: kept from v2 {min(len(kept), n_old)}, top-up from v2 pool {max(n_old - len(kept), 0)}, newly eligible {k_new}')
sample = pd.concat(parts).reset_index(drop=True)
# Checks: every sampled account meets every rule.
assert sample['username_key'].is_unique
assert (sample['first_action_utc'] > TFA_CREATED_UTC).all() and (sample['first_action_utc'] < ACCUSED_BEFORE_UTC).all()
assert (sample['first_mention_adj_utc'] < ACCUSED_BEFORE_UTC).all() and (sample['first_action_utc'] < sample['first_mention_adj_utc']).all()
assert not (sample['edit_lag_h'] > MAX_EDIT_LAG_H).any()
print(sample['account_status'].value_counts().to_string())

## 3. Attach Arctic Shift metadata and PRAW profiles

In [ ]:
# Attach Arctic Shift metadata (prefix arctic_) and PRAW profile fields (prefix praw_) to the sample.
# Image URL columns are dropped; account_status comes from the sample, not the profile table.
profiles = pd.read_parquet(os.path.join(DATA_DIR, 'account_profiles_full.parquet'))
meta_cols = (meta.drop(columns=['fetch_error', 'earliest_post_at', 'earliest_comment_at'])
                 .add_prefix('arctic_').rename(columns={'arctic_username_key': 'username_key'}))
profile_cols = (profiles.drop(columns=['account_status', 'icon_img', 'snoovatar_img'])
                        .add_prefix('praw_').rename(columns={'praw_username_key': 'username_key'}))
sample_accounts = (sample.merge(meta_cols, on='username_key', how='left', validate='one_to_one')
                         .merge(profile_cols, on='username_key', how='left', validate='one_to_one'))
print(sample_accounts.shape)

## 4. The sample's stored history

In [ ]:
# Load the stored posts and comments of the sampled accounts only.
keys = set(sample['username_key'])
sample_posts = read_history('posts', keys=keys)
sample_comments = read_history('comments', keys=keys)
print(f'posts {len(sample_posts):,}   comments {len(sample_comments):,}')

# Rows stored per account, by status: a check that every sampled account has history.
counts = pd.DataFrame({'posts': sample_posts['outer_key'].value_counts(),
                       'comments': sample_comments['outer_key'].value_counts()}).reindex(sorted(keys)).fillna(0).astype(int)
counts['account_status'] = counts.index.map(sample.set_index('username_key')['account_status'])
counts.groupby('account_status')[['posts', 'comments']].agg(['sum', 'median', 'mean', 'max'])

## 5. Save

In [ ]:
# Save the eligible pool, the sample accounts, and their posts and comments (never overwrites).
save_new(eligible[['username_key', 'account_status', 'first_mention_utc', 'accusation_post_edited_utc', 'first_mention_adj_utc',
                  'first_action_utc', 'days_to_accused', 'history_truncated', 'in_v2_pool']],
         'eligible_pool.parquet')
save_new(sample_accounts, 'sample_accounts.parquet')
save_new(sample_posts, 'sample_posts.parquet')
save_new(sample_comments, 'sample_comments.parquet')
# v2 sampled accounts that were dropped, with the rule they failed.
save_new(dropped, 'dropped_from_v2.parquet')